# Human review — reviewer 2

Use this notebook if you are **reviewer 2**. Your ID is already set, and your answers save
to `reviewer_2.csv`.

Both reviewers read the same 100 articles and rate each model's label separately: **200 items
per person**. Work independently and wait until both people finish before comparing answers.

Run cells with `Shift+Enter`. Load your file, read the rubric, then repeat the article-display
and answer-entry cells for each item. Use the progress cell when you want to pause or finish.
Running all cells at once will not complete the ratings for you.

## Before you start

The coordinator needs to finalize both models and their labels, then run
[pipeline_comparison.ipynb](../comparison/pipeline_comparison.ipynb) to prepare the review packet.
If those steps are unfinished, this notebook shows a pending message.

Your folder needs:

- `review_packet.csv`: article text and model assignments.
- `review_freeze.json` and `review_status.json`: records used to check the packet.
- `reviewer_2.csv`: your answer file.

You do not need the other person's CSV. Keep the articles, model assignments, and labels fixed
during review. The coordinator can use [review_coordinator.ipynb](review_coordinator.ipynb)
to check progress and summarize the results later.

Start Jupyter with `uv run jupyter lab`. Before entering ratings, save a personal copy as
`reviewer_2_workbook.ipynb` inside the packet's `human_review` folder. Git ignores that
working copy, which may contain article text and your answers. Keep the notebook in the repository
as a blank template.

The default packet folder is
`final-project/code/modern-methods/Results/comparison/<selected-run>/human_review/`.

In [1]:
from pathlib import Path
import html
import json
import sys
import pandas as pd
from IPython.display import HTML, display

# Locate the project whether Jupyter opened from the repository or notebook folder.
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
ROOT = REPO / "final-project/code/modern-methods"
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT.parent))
from src.reviewer import load_session, choose_item, save_answer


## Step 1 — load your file

Leave `REVIEWER = "reviewer_2"` unchanged. Normally, leave `PACKET_FOLDER = None` too;
the code finds the selected model's review folder. If the coordinator gave you a different
folder, enter its absolute path.

This step loads your existing answers and checks that they match the packet. Missing files show
as pending. If there is an error about changed labels, article IDs, or invalid answers, ask the
coordinator to check it before continuing.

In [2]:
# Preconfigured for this notebook. Leave this reviewer ID unchanged.
REVIEWER = "reviewer_2"

# OPTIONAL: only change this if the coordinator gave you a separate packet folder.
PACKET_FOLDER = None              # Example: Path("/absolute/path/to/human_review")

SESSION = None
CURRENT_ITEM = None
REVIEW_FOLDER = None

if PACKET_FOLDER is not None:
    REVIEW_FOLDER = Path(PACKET_FOLDER).expanduser().resolve()
else:
    selection_file = ROOT / "Results/modern/selection.json"
    selection = json.loads(selection_file.read_text()) if selection_file.exists() else {}
    if selection.get("run_id"):
        REVIEW_FOLDER = ROOT / "Results/comparison" / selection["run_id"] / "human_review"

if REVIEW_FOLDER is None:
    print("Pending: the coordinator must finish model selection, labels, and pipeline_comparison.ipynb's review-packet step.")
else:
    required = ["review_packet.csv", "review_freeze.json", "review_status.json", f"{REVIEWER}.csv"]
    missing = [name for name in required if not (REVIEW_FOLDER / name).exists()]
    if missing:
        print("Pending review files:", ", ".join(missing))
        print("Ask the coordinator to supply the frozen packet and your own answer file.")
    else:
        SESSION = load_session(REVIEW_FOLDER, REVIEWER)
        print(f"Opened {REVIEWER}: {int(SESSION['complete'].sum())}/{len(SESSION['packet'])} items complete.")
        print("Your answers are saved in:", SESSION["path"])

Pending: the coordinator must finish model selection, labels, and pipeline_comparison.ipynb's review-packet step.


## Step 2 — read the rating guide

Rate **how well the topic label describes the article**. Read the full text, not just the title.
The score is not about whether you agree with the author or whether the article supports
autonomous vehicles.

| Score | Meaning |
| --- | --- |
| **1** | The label is unrelated or misleading. |
| **2** | The label fits only a passing mention or small part of the article. |
| **3** | The label fits a meaningful part, but misses substantial content. |
| **4** | The label describes a main theme, with minor omissions or imprecision. |
| **5** | The label clearly and specifically describes a central theme. |

Fill in these fields:

| Field | What to enter |
| --- | --- |
| `FIT_RATING` | A whole number from 1–5. Use `None` for unassigned articles (topic −1). |
| `IRRELEVANT` | `True` if the article is not really about autonomous vehicles; otherwise `False`. A passing mention may not be enough to make it relevant. |
| `AMBIGUOUS_LABEL` | `True` if the label is unclear, too broad, or has several possible meanings; otherwise `False`. A clear but incorrect label can be a poor fit without being ambiguous. Use `False` for topic −1, which has no label to assess. |
| `NOTES` | A brief explanation, especially for low ratings, unclear labels, irrelevant articles, or unassigned articles. Use your own words rather than copying long passages. |

Keep relevance and fit separate: an article can be irrelevant to autonomous driving but still
fit its assigned topic. Judge each model's label separately; the scores do not have to match.
For topic −1, leave the fit rating blank and complete the flags. Do not give it a 1 just because
it was unassigned.

Use your own reading to score each item, without AI-generated scores, model metrics, or the other
reviewer's answers. You can see the model names, so this review is not blinded. If you remember
an article from earlier topic-selection work, note that and tell the coordinator.

## Step 3 — read the next article

Leave `ITEM_NUMBER = None` to show the next unfinished item. To revisit an answer, enter its item
number, starting at 1. The display includes any answer you already saved. Scroll through the box
to read the full article.

After saving in Step 4, run this cell again to move to the next item.

In [3]:
# Normally leave this as None. For a correction, enter the displayed item number.
ITEM_NUMBER = None
CURRENT_ITEM = None

if SESSION is None:
    print("Finish Step 1 before displaying an article.")
else:
    # Reload your own CSV so an interrupted session resumes from saved answers.
    SESSION = load_session(SESSION["folder"], SESSION["reviewer"])
    CURRENT_ITEM = choose_item(SESSION, ITEM_NUMBER)
    if CURRENT_ITEM is None:
        print("All of your items are complete. Continue to Step 5.")
    else:
        article = SESSION["packet"].iloc[CURRENT_ITEM]
        print(f"ITEM {CURRENT_ITEM + 1} of {len(SESSION['packet'])} — copy this item number into ITEM_TO_SAVE below.")
        display(pd.Series({"article_id": article.article_id, "model": article.model,
                           "topic_id": article.topic_id, "assigned_label": article.label,
                           "title": article.title}, name="Review this assignment"))
        if article.topic_id == -1:
            print("UNASSIGNED: leave FIT_RATING = None; complete the qualitative flags and notes.")
        # Escape article text before rendering: article HTML must be displayed as text.
        display(HTML('<div style="white-space:pre-wrap;max-height:650px;overflow:auto;'
                     'padding:18px;border:1px solid #aaa;line-height:1.6">'
                     + html.escape(str(article.text)) + '</div>'))
        display(SESSION["answers"].loc[CURRENT_ITEM, ["fit_rating", "irrelevant", "ambiguous_label", "notes"]].rename("Your saved answers"))

Finish Step 1 before displaying an article.


## Step 4 — enter and save your answer

Fill in the values below for the article you just read. Use capitalized `True`, `False`, and
`None`, and put notes in quotes. Copy the displayed item number into `ITEM_TO_SAVE`, set
`SAVE_RATING = True`, and run the cell.

The item number helps prevent saving an answer for the wrong article. If the packet or CSV
changed in another session, reload before continuing. To correct an answer you already saved,
display that item first and set `ALLOW_UPDATE = True`.

For each item: **display → read → update every answer field and item number → save**.

In [4]:
# FILL THESE IN for the displayed item; no answers are supplied automatically.
ITEM_TO_SAVE = None          # Copy the displayed ITEM number here (e.g., 1).
FIT_RATING = None            # 1, 2, 3, 4, or 5. Keep None only for an unassigned item.
IRRELEVANT = None            # Replace with True or False.
AMBIGUOUS_LABEL = None       # Replace with True or False.
NOTES = ""                  # Write your reason or any uncertainty in your own words.

SAVE_RATING = False          # Change to True when this item's answers are ready to save.
ALLOW_UPDATE = False         # True only when intentionally correcting existing answers.

if not SAVE_RATING:
    print("Nothing saved. Read the item and fill in the fields before enabling SAVE_RATING.")
elif SESSION is None or CURRENT_ITEM is None:
    print("Nothing saved. Display an item in Step 3 first.")
elif type(ITEM_TO_SAVE) is not int or ITEM_TO_SAVE != CURRENT_ITEM + 1:
    raise ValueError("ITEM_TO_SAVE must match the displayed item's number. Check it before saving.")
else:
    SESSION = save_answer(SESSION, CURRENT_ITEM, FIT_RATING, IRRELEVANT, AMBIGUOUS_LABEL,
                          notes=NOTES, allow_update=ALLOW_UPDATE)
    print(f"Saved item {ITEM_TO_SAVE} to {SESSION['reviewer']}.csv.")
    print(f"Progress: {int(SESSION['complete'].sum())}/{len(SESSION['packet'])} items complete.")
    print("Rerun Step 3 to display the next item, then update all answer fields.")
    CURRENT_ITEM = None      # A new display is required before another save.
SAVE_RATING = False

Nothing saved. Read the item and fill in the fields before enabling SAVE_RATING.


## Step 5 — check progress or finish

This cell checks your file and lists unfinished items. Both flags are required, including for
unassigned articles. Notes are optional, but helpful for difficult cases.

To take a break, save your current answer and close Jupyter. To resume, reopen your notebook,
run the setup cell and Steps 1–3, and leave `ITEM_NUMBER = None`.

When finished, send `reviewer_2.csv` to the coordinator. Keep your original independent
ratings when discussing disagreements afterward.

Your working notebook may contain article text in its outputs and ratings typed into code cells.
Share the CSV rather than that notebook. To make a blank notebook copy, reset the answer fields
and save switches, then restart the kernel and clear outputs. Clearing outputs alone does not
remove answers typed into cells.

In [5]:
if SESSION is None:
    print("Load your assigned review file in Step 1 first.")
else:
    SESSION = load_session(SESSION["folder"], SESSION["reviewer"])
    total = len(SESSION["packet"])
    finished = int(SESSION["complete"].sum())
    print(f"{SESSION['reviewer']}: {finished}/{total} items complete ({finished / total:.0%}).")
    remaining = SESSION["answers"].loc[~SESSION["complete"], ["article_id", "model", "topic_id"]].copy()
    remaining.insert(0, "item_number", remaining.index + 1)
    if remaining.empty:
        print("Your file is complete and valid. Tell the coordinator; keep your independent ratings unchanged.")
    else:
        print("Next unfinished items (first 20):")
        display(remaining.head(20))
    print("Your saved file:", SESSION["path"])

Load your assigned review file in Step 1 first.
